# EquiSense Backtest Diagnostics

Этот ноутбук разбирает поведение стратегии из `notebooks/results/backtest_curves.csv`: динамика equity, drawdown, помесячная доходность, риск-профиль.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")
ROOT = Path.cwd().resolve().parents[0]
res = ROOT / "notebooks" / "results"
df = pd.read_csv(res / "backtest_curves.csv", parse_dates=["date"]).sort_values("date")
df["strategy_dd"] = df["strategy_curve"] / df["strategy_curve"].cummax() - 1
df["bh_dd"] = df["buy_hold_equally_weighted"] / df["buy_hold_equally_weighted"].cummax() - 1
monthly = df.set_index("date")[["strategy_ret", "ret_1d"]].resample("M").sum().rename(columns={"strategy_ret":"strategy","ret_1d":"buy_hold"})
ann_vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
ann_vol_bh = df["ret_1d"].std() * np.sqrt(252)
ann_ret_strategy = (1 + df["strategy_ret"].mean())**252 - 1
ann_ret_bh = (1 + df["ret_1d"].mean())**252 - 1
print({"annual_ret_strategy": round(float(ann_ret_strategy),4), "annual_ret_bh": round(float(ann_ret_bh),4), "annual_vol_strategy": round(float(ann_vol_strategy),4), "annual_vol_bh": round(float(ann_vol_bh),4)})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.lineplot(data=df, x="date", y="strategy_curve", ax=axes[0,0], label="Strategy")
sns.lineplot(data=df, x="date", y="buy_hold_equally_weighted", ax=axes[0,0], label="Buy&Hold")
axes[0,0].set_title("Equity curves")

sns.lineplot(data=df, x="date", y="strategy_dd", ax=axes[0,1], label="Strategy DD")
sns.lineplot(data=df, x="date", y="bh_dd", ax=axes[0,1], label="Buy&Hold DD")
axes[0,1].set_title("Drawdown")

sns.heatmap(monthly.T, cmap="RdYlGn", center=0, ax=axes[1,0])
axes[1,0].set_title("Monthly return heatmap")

sns.histplot(df["strategy_ret"], bins=80, kde=True, ax=axes[1,1], color="#4c72b0")
axes[1,1].set_title("Strategy daily returns distribution")

plt.tight_layout()

## Выводы

- Диагностика drawdown показывает, в какие периоды стратегия наиболее уязвима.
- Помесячная теплокарта помогает быстро увидеть кластеры прибыльных/убыточных режимов.
- Перед продом стоит добавить risk overlay: ограничение позиций при росте rolling-volatility.